# End-to-End ARC-05 Workflow

This notebook demonstrates the package workflow from a translational orbital state to a sampled trajectory and diagnostics, using the public `OrbitalState`, `SimulationConfig`, and `run_simulation` interfaces. It does not duplicate the equations or integrator implementation.

The example uses a near-circular Earth orbit, ECI coordinates, metres, seconds, metre-per-second velocities, Earth point-mass gravity, and fixed-step RK4. Collision detection is a sampled threshold check; any between-sample event time/state uses linear radial interpolation and is only an approximation.

This educational two-body model omits perturbations such as drag, J2, thrust, and third-body gravity. Its output is not mission-grade validation, certification, or operational safety guidance.

In [1]:
import sys as _notebook_sys
from pathlib import Path as _NotebookPath

_notebook_cwd = _NotebookPath.cwd()
_notebook_repo_root = next(
    (
        candidate
        for candidate in (_notebook_cwd, *_notebook_cwd.parents)
        if (candidate / "pyproject.toml").is_file()
        and (candidate / "src" / "quantum_space_agi").is_dir()
    ),
    None,
)
if _notebook_repo_root is None:
    raise RuntimeError(
        f"Could not locate repository root from notebook working directory {_notebook_cwd}; "
        "open the notebook from within the repository."
    )
_notebook_src_path = str(_notebook_repo_root / "src")
if _notebook_src_path not in _notebook_sys.path:
    _notebook_sys.path.insert(0, _notebook_src_path)

import quantum_space_agi as _notebook_package
if _NotebookPath(_notebook_package.__file__).resolve().parent != (
    _notebook_repo_root / "src" / "quantum_space_agi"
).resolve():
    raise ImportError("quantum_space_agi did not resolve from this repository's src directory")

from pathlib import Path
import sys

repo_root = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "pyproject.toml").is_file()
        and (candidate / "src" / "quantum_space_agi").is_dir()
    ),
    None,
)
if repo_root is None:
    raise RuntimeError("Could not locate the repository root from the notebook working directory")
src_path = str(repo_root / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

import numpy as np
from quantum_space_agi.simulation import OrbitalState, SimulationConfig, run_simulation

In [2]:
initial_state = OrbitalState(
    position_m=(7_000_000.0, 0.0, 0.0),
    velocity_mps=(0.0, 7_500.0, 0.0),
    time_s=0.0,
)
config = SimulationConfig(
    initial_position_m=initial_state.position_m,
    initial_velocity_mps=initial_state.velocity_mps,
    simulation_duration_s=600.0,
    time_step_s=10.0,
    collision_radius_m=6_378_137.0,
)
print("Initial state [x, y, z, vx, vy, vz]:", initial_state.to_array())
print("Frame / integrator:", config.reference_frame, "/", config.integrator)
print("Duration / timestep (s):", config.simulation_duration_s, "/", config.time_step_s)

Initial state [x, y, z, vx, vy, vz]: [7000000.       0.       0.       0.    7500.       0.]
Frame / integrator: ECI / RK4
Duration / timestep (s): 600.0 / 10.0


In [3]:
result = run_simulation(config, initial_time_s=initial_state.time_s)
times = result["times_s"]
states = result["states"]
diagnostics = result["diagnostics"]

assert states.shape == (times.size, 6)
assert times[0] == initial_state.time_s
assert times[-1] == config.simulation_duration_s
assert np.all(np.isfinite(times))
assert np.all(np.isfinite(states))

print("Returned fields:", sorted(result))
print("Model:", result["model"])
print("Trajectory arrays:", times.shape, states.shape)
print("Final position (m):", states[-1, :3])
print("Final velocity (m/s):", states[-1, 3:])
print("Final radius / speed:", diagnostics["radius_m"][-1], "m /", diagnostics["speed_m_s"][-1], "m/s")
print("Maximum absolute energy drift (m^2/s^2):", diagnostics["max_abs_energy_drift_m2_s2"])
print("Maximum absolute angular-momentum drift (m^2/s):", diagnostics["max_abs_angular_momentum_drift_m2_s"])
print("Collision method / time:", result["collision_method"], "/", result["collision_time_s"])

Returned fields: ['collision_index', 'collision_method', 'collision_radius_m', 'collision_state', 'collision_time_s', 'diagnostics', 'model', 'states', 'times_s']
Model: {'reference_frame': 'ECI', 'integrator': 'RK4', 'dynamics': 'Earth point-mass Newtonian two-body', 'time_unit': 's', 'position_unit': 'm', 'velocity_unit': 'm/s'}
Trajectory arrays: (61,) (61, 6)
Final position (m): [5584427.92115119 4192052.29840607       0.        ]
Final velocity (m/s): [-4558.02975893  5979.5741559      0.        ]
Final radius / speed: 6982774.354016075 m / 7518.706163244238 m/s
Maximum absolute energy drift (m^2/s^2): 4.485994577407837e-05
Maximum absolute angular-momentum drift (m^2/s): 0.068603515625
Collision method / time: None / None
